# TBsim demonstration notebook

TBsim is an agent-based TB transmission and disease progression simulator built on Starsim. It models the LSHTM spectrum-of-disease natural history (susceptible -> infection -> non-infectious -> asymptomatic -> symptomatic -> treatment/cleared/dead) and supports interventions such as diagnostics, treatment delivery, vaccination, and nutrition/comorbidity effects within configurable populations and networks. This notebook demonstrates how to configure TBsim/Starsim calls to explore baseline dynamics, care cascades, prevention, comorbidities, drug resistance, and scenario comparisons.

## Research basis

This section summarizes the research document as a setting for the TBsim demonstration. It describes a population of TB-prone individuals over a multi-year horizon with interventions commonly used in TB programs: health-seeking behavior, screening, confirmatory testing, and standard TB treatment delivery. Assumptions and limitations are noted as placeholders where the document is silent or where TBsim defaults are used.

Parameter table (values drawn from the research doc or marked as placeholders):

| Parameter | Value | Units | Source |
|---|---:|---|---|
| n_agents | 3000 | agents | research doc placeholder |
| start | 2000-01-01 | date | TBsim default |
| stop | 2020-12-31 | date | TBsim default |
| dt | 7 days | days | TBsim default |
| beta | 0.2 per year | 1/year | TBsim default |
| init_prev | 0.05 | proportion | placeholder - not in doc |
| health-seeking baseline | 0.1 per day | per day | research doc |
| screening coverage | 0.9 | proportion | research doc |
| xpert coverage | 0.8 | proportion | research doc |
| dx to treatment (DOTS) coverage | 0.85 | proportion | research doc |
| BCG coverage (under-5) | 0.8 | proportion | placeholder - not in doc |
| TPTHousehold coverage | 0.8 | proportion | placeholder - not in doc |
| BetaByYear (transmission change) | year 2010, x_beta 0.7 | - | placeholder - not in doc |
| HIV init_prev | 0.02 | proportion | placeholder - not in doc |
| Malnutrition | present | - | placeholder - not in doc |
    

Notes:
- Values labeled as placeholders are not specified in the research document and are shown for demonstration purposes only.
- All other values use TBsim defaults where not specified in the research doc.

In [ ]:
!pip install -q git+https://github.com/starsimhub/tbsim.git

import numpy as np
import sciris as sc
import starsim as ss
import tbsim
import matplotlib.pyplot as plt
import pandas as pd

print('tbsim version:', tbsim.__version__)
print('starsim version:', ss.__version__)

In [ ]:
# Shared base parameters for TBsim simulations
# Fast configuration suitable for Colab demonstration
BASE = dict(
    n_agents=3000,
    start=ss.date('2000-01-01'),
    stop=ss.date('2020-12-31'),
    dt=ss.days(7),
    rand_seed=1,
    verbose=0,
    beta=ss.peryear(0.2),
    init_prev=ss.bernoulli(p=0.05),
)

## Baseline natural history
Baseline TB transmission and state progression using TBsim defaults where not specified in the document. This run uses a simple random network and the base TB model without additional interventions beyond the usual TB disease progression.

In [ ]:
# Baseline natural history simulation
networks = [ss.RandomNet(pars=dict(n_contacts=ss.poisson(lam=8), dur=0))]

sim = tbsim.Sim(**BASE, networks=networks, analyzers=[tbsim.DwtAnalyzer(scenario_name='baseline')])
sim.run()
tb = sim.get_tb()
timevec = sim.timevec

# Plot prevalence and incidence over time
plt.figure(figsize=(8,4))
plt.plot(timevec, sim.results.tb_prevalence.values, label='tb_prevalence')
plt.plot(timevec, sim.results.tb_incidence_kpy.values, label='tb_incidence_kpy')
plt.xlabel('Time (years)')
plt.ylabel('Rate / Prevalence')
plt.legend()
plt.title('Baseline TB prevalence and incidence over time')
plt.tight_layout()
plt.show()

summary = {
    'scenario': ['Baseline'],
    'final_prevalence': [float(sim.results.tb_prevalence.values[-1])],
    'cumulative_infections': [float(sim.results.tb_cum_infections.values[-1])],
    'tb_deaths': [float(sim.results.tb_deaths_ppy.values[-1])]        
}
pd.DataFrame(summary)

## Care cascade
This section adds a care cascade: health-seeking behavior, screening, confirmatory testing, and treatment delivery. The cascade provides a basic demonstration of how TBcare interventions alter the trajectory compared to baseline.

In [ ]:
# Define care cascade interventions
itvs = [
    tbsim.HealthSeekingBehavior(pars=dict(initial_care_seeking_rate=ss.perday(0.1))),
    tbsim.DxDelivery(name='screen', product=tbsim.CAD(), coverage=0.9, result_state='screen_positive'),
    tbsim.DxDelivery(name='confirm', product=tbsim.Xpert(), coverage=0.8, eligibility=lambda sim: sim.people.screen.screen_positive.uids, result_state='diagnosed'),
    tbsim.TxDelivery(product=tbsim.DOTS()),
]

sim_cascade = tbsim.Sim(**BASE, networks=networks, interventions=itvs, analyzers=[tbsim.DwtAnalyzer(scenario_name='cascade')])
sim_cascade.run()

plt.figure(figsize=(8,4))
plt.plot(sim_cascade.timevec, sim_cascade.results.tb_prevalence.values, label='tb_prevalence')
plt.plot(sim_cascade.timevec, sim_cascade.results.tb_incidence_kpy.values, label='tb_incidence_kpy')
plt.xlabel('Time (years)')
plt.ylabel('Rate / Prevalence')
plt.legend()
plt.title('Care cascade: baseline vs cascade (placeholder)')
plt.tight_layout()
plt.show()

cascade_summary = {
    'scenario': ['Cascade'],
    'final_prevalence': [float(sim_cascade.results.tb_prevalence.values[-1])],
    'cumulative_infections': [float(sim_cascade.results.tb_cum_infections.values[-1])],
    'tb_deaths': [float(sim_cascade.results.tb_deaths_ppy.values[-1])]        
}
pd.DataFrame(cascade_summary)

## Prevention
Prevention components include BCG vaccination, household Tuberculosis Preventive Therapy (TPT) with a household network, and a BetaByYear transmission change to illustrate calendar-driven shifts in transmission.

In [ ]:
# Define networks with a household network to enable TPT within households
def hh_net(n=400):
    ages = [sc.strjoin(np.random.randint(1, 70, np.random.randint(2, 6))) for _ in range(n)]
    return ss.library.HouseholdNet(dhs_data=sc.dataframe(hh_id=np.arange(n), ages=ages), dynamic=False)

networks_p = [ss.RandomNet(pars=dict(n_contacts=ss.poisson(lam=5), dur=0)), hh_net()]

itvs_p = [
    tbsim.BCGRoutine(pars=dict(coverage=0.8, age_range=[0, 5])),
    tbsim.TPTHousehold(pars=dict(coverage=0.8)),
    tbsim.BetaByYear(pars=dict(years=[2010], x_beta=[0.7]))
]

sim_prevent = tbsim.Sim(**BASE, networks=networks_p, interventions=itvs_p, analyzers=[tbsim.DwtAnalyzer(scenario_name='prevention')])
sim_prevent.run()

plt.figure(figsize=(8,4))
plt.plot(sim_prevent.timevec, sim_prevent.results.tb_prevalence.values, label='tb_prevalence')
plt.xlabel('Time (years)')
plt.ylabel('Prevalence')
plt.legend()
plt.title('Prevention scenario: BCG, TPT, BetaByYear')
plt.tight_layout()
plt.show()

## Comorbidities
HIV and malnutrition are added as comorbidities with their respective connectors to TB. HIV interventions and nutrition connectors accompany a TB-HIV interaction mechanism.

In [ ]:
# HIV and malnutrition comorbidity setup (placeholder connectors)
hiv = tbsim.HIV(pars=dict(init_prev=ss.bernoulli(0.02), init_onart=ss.bernoulli(0.01)))
hivint = tbsim.HivInterventions(pars=dict(use_prevalence=True, use_art=True, prevalence=0.2, percent_on_ART=0.5, min_age=15, max_age=60, start=ss.date('2005-01-01'), stop=ss.date('2020-12-31')))
sim_hiv = tbsim.Sim(**BASE, diseases=[hiv, tbsim.Malnutrition()], interventions=[hivint], connectors=[tbsim.TB_HIV_Connector(), tbsim.TB_Nutrition_Connector()])
sim_hiv.run()

print('HIV + TB-comorbidity run completed. Final TB prevalence:', float(sim_hiv.results.tb_prevalence.values[-1]))


## Drug resistance
Demonstration of a resistance-aware TB model by creating a TBResistant TB strain and a corresponding treatment product with strain-aware efficacy. This cell demonstrates how to access ResistanceStats through TBsim.

In [ ]:
tbr = tbsim.TBResistant(drugs=['RIF'], rel_fitness={'RIF': 0.9}, beta=ss.permonth(0.35), init_prev=ss.bernoulli(p=0.12))
tx = tbsim.TxDeliveryR(rate_sym=ss.peryear(1.0), product=tbsim.TxR(strains=tbr.strains, base_efficacy=0.8, resist_penalty={'RIF': 0.2}))
sim_res = tbsim.Sim(tb_model=tbr, n_agents=2000, demographics=[], networks=ss.RandomNet(pars=dict(n_contacts=ss.poisson(lam=8), dur=0)), interventions=[tx], analyzers=[tbsim.ResistanceStats()], dt=ss.days(30), start=ss.date('2000-01-01'), stop=ss.date('2015-01-01'), verbose=0)
sim_res.run()
print('Resistance stats available:', sim_res.results.ResistanceStats if hasattr(sim_res.results, 'ResistanceStats') else 'not available')


## Scenario comparison
Compare baseline, cascade, and prevention scenarios. This section runs multiple simulations (one per scenario) and plots selected outputs side-by-side. A simple summary table is produced with final prevalence, cumulative infections, and TB deaths.

In [ ]:
# Run multiple scenarios and compare
scenarios = {}

# Baseline (reuse previous sim object if desired, otherwise recreate)
sim1 = sim  # from Baseline cell
scenarios['Baseline'] = sim1.results.flatten()

sim2 = sim_cascade  # Cascade
scenarios['Cascade'] = sim2.results.flatten()

sim3 = sim_prevent  # Prevention
scenarios['Prevention'] = sim3.results.flatten()

rows = []
for name, res in scenarios.items():
    rows.append({
        'Scenario': name,
        'Final Prevalence': float(res.tb_prevalence.values[-1]),
        'Cumulative Infections': float(res.tb_cum_infections.values[-1]),
        'TB Deaths per Year (ppy)': float(res.tb_deaths_ppy.values[-1]) if hasattr(res, 'tb_deaths_ppy') else np.nan
    }))
pd.DataFrame(rows)

## Dwell-time analysis
Dwell-time (DwtAnalyzer) results from the cascade run provide distribution of time spent in states along the care cascade. This section simply demonstrates accessing the DwtAnalyzer results.

In [ ]:
if hasattr(sim_cascade.results, 'dwt'):
    dwt = sim_cascade.results.dwt
    print('Dwell-time stats available for cascade:')
    print(dwt.summary())
else:
    print('Dwell-time analyzer not present in cascade results (placeholder).')

## Sanity checks
Simple assertions to ensure results are finite and non-negative, and prevalence lies within [0,1].

In [ ]:
def asserts_ok(simobj):
    tbprev = simobj.results.tb_prevalence.values
    assert np.isfinite(tbprev).all(), 'tb_prevalence contains non-finite values'
    assert (tbprev >= 0).all() and (tbprev <= 1.0).all(), 'tb_prevalence outside [0,1]'
    return True

asserts_ok(sim)
asserts_ok(sim_cascade)
asserts_ok(sim_prevent)

## Interpretation and limitations
- This notebook demonstrates how to configure and run TBsim/Starsim components using the API patterns from the TBsim 0.10.3 / Starsim 3.7 stack.
- Outputs are illustrative and intended for demonstration; many parameter values are placeholders where the research document did not specify exact values. This is not a validated clinical model.
- All modules, interventions, and connectors used here are drawn from TBsim/Starsim APIs and not reimplemented.